# Task 2 — F-cls: Qwen2.5-Coder-7B + QLoRA + head 10 nhãn (Kaggle T4×2)

Fine-tune phân loại đa nhãn (xem `docs/task2_finetune_plan.md`). Model đọc đề + code + kết quả compile/chạy rồi cho xác suất của 10 nhãn.
Ngưỡng từng nhãn được chọn theo out-of-fold trên 32 bài gold, sau đó áp luật gold. Public chỉ dùng để dự đoán.

**Chuẩn bị**
1. Settings: Accelerator **GPU T4 x2**, Internet **On**.
2. Secrets (Add-ons → Secrets):
   - `PUBLIC_TEST_DRIVE_ID`: file id Drive của bộ dữ liệu cuộc thi.
   - `SILVER_DRIVE_ID`: file id Drive của **`generate-dataset_v3.zip`**.
   - Tuỳ chọn: `GITHUB_TOKEN`, `HF_TOKEN`.
   - Không dán id vào notebook rồi commit: repo là public.
3. `BRANCH = 'task-2-e2-signals'`.

**Chạy:** *Save Version → Save & Run All (Commit)*. Notebook sinh bộ test E2 (Qwen 7B gốc đọc đề, ~1 giờ, lưu ở `/kaggle/working/testgen`), dựng dữ liệu kèm tóm tắt chạy test, rồi train 3 run:
- `EX01` và `EX02` (2 fold, chạy song song trên 2 GPU) để đo out-of-fold trên gold;
- `all` (model final) để dự đoán public.

**Khi bị timeout / dừng giữa chừng.** Mỗi run lưu checkpoint (adapter + optimizer + scheduler) mỗi `save_steps` step vào `/kaggle/working/cls/<run>/checkpoints/`.
Ngoài ra script tự **dừng sạch trước giới hạn phiên** (`SESSION_HOURS`), để notebook kết thúc bình thường và Kaggle lưu `/kaggle/working`.
Run chưa xong thì `status.json` = `paused`. Để chạy tiếp:
1. Mở notebook → **Add Input** → *Your Work / Notebook Output* → chọn version vừa chạy của chính notebook này (được mount vào `/kaggle/input/...`).
2. *Save & Run All* lại. Script tìm checkpoint mới nhất trong `/kaggle/input` và resume đúng step đó. Run đã `done` thì bỏ qua.
- Tuỳ chọn chống crash cứng (mất điện, OOM): đặt `HUB_REPO = '<user>/task2-cls-ckpt'` + secret `HF_TOKEN`. Checkpoint cuối được đẩy lên HF Hub (repo private) sau mỗi lần lưu và được kéo về khi resume.

**Output** (`/kaggle/working`):
- `task2_predictions_exec.json` (**nộp bài**): E2 = Qwen 7B sinh test từ đề, chạy bài trên test + luật giảng viên, nguyên nhân lỗi lấy từ F-cls;
- `task2_predictions.json`: F-cls thuần;
- `cls/exec2_<tag>/`: `report.md`, `signals_public.jsonl` (trạng thái chạy test từng bài); bộ test sinh ra ở `outputs/task2/_testgen/`;
- `cls/<name>_report/`: `report.md`, `metrics.json` (fixed / cross-fold tuned / in-sample), `oof_gold.json`;
- `cls/<run>/`: `train_log.jsonl`, `run_info.json` (phiên bản model, thư viện, GPU, VRAM đỉnh), `adapter/`.


In [ ]:
import time
T0 = time.time()               # session start: the training deadline is counted from here
REPO = 'github.com/tiendinhquang2104/llm-final-project.git'
BRANCH = 'task-2-e2-signals'
DRIVE_ID = ''                  # or secret PUBLIC_TEST_DRIVE_ID (never commit the id: the repo is public)
SILVER_DRIVE_ID = ''           # generate-dataset_v3.zip, or secret SILVER_DRIVE_ID
WORK = '/kaggle/tmp'           # repo + data (not saved with the version)
OUT = '/kaggle/working/cls'    # runs, checkpoints, reports (saved with the version -> resume next time)
CONFIG = 'configs/task2/cls_qwen7b.yaml'
SESSION_HOURS = 11.5           # commit sessions stop at 12h; interactive sessions: set what is left
HUB_REPO = ''                  # e.g. 'myuser/task2-cls-ckpt' (+ secret HF_TOKEN): mirror of the last checkpoint
TAG = 'sig'                    # suffix of the run folders: new config -> new tag (old runs are 'done' and would be skipped)
EPOCHS = None                  # override train.epochs (None = config)
LR = None                      # override train.lr
SMOKE = False                  # True: 16 training rows, 1 epoch (checks the whole pipeline in ~15 min)


In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ''
token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
os.makedirs(WORK, exist_ok=True)
REPO_DIR = f'{WORK}/repo'
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone failed — check REPO/BRANCH/GITHUB_TOKEN') from None  # never print the token URL
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'], check=True)
os.chdir(REPO_DIR)
if secret('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret('HF_TOKEN')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q gdown 'transformers>=4.51,<5' 'peft>=0.13' 'bitsandbytes>=0.45' accelerate safetensors scikit-learn pyyaml
!g++ --version | head -1
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import gdown, shutil
drive_id = DRIVE_ID or secret('PUBLIC_TEST_DRIVE_ID')
assert drive_id, 'Set DRIVE_ID (or the PUBLIC_TEST_DRIVE_ID secret)'
if not os.path.exists('data/raw/sample_dataset'):
    os.makedirs(f'{WORK}/download', exist_ok=True)
    assert gdown.download(id=drive_id, output=f'{WORK}/download/data.zip', quiet=False), 'gdown failed (public data)'
    !python scripts/unpack_data.py {WORK}/download --dest data/raw
SYNTH_ZIP = 'data/synth/v1/generate-dataset_v3.zip'   # = data.synth_zip in the config
if not os.path.exists(SYNTH_ZIP):
    found = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs if f == 'generate-dataset_v3.zip']
    os.makedirs(os.path.dirname(SYNTH_ZIP), exist_ok=True)
    if found:
        shutil.copy(found[0], SYNTH_ZIP)
    else:
        sid = SILVER_DRIVE_ID or secret('SILVER_DRIVE_ID')
        assert sid, 'Set SILVER_DRIVE_ID (generate-dataset_v3.zip) or add the zip as a Kaggle input'
        assert gdown.download(id=sid, output=SYNTH_ZIP, quiet=False), 'gdown failed (generate-dataset_v3.zip)'
!ls -la data/raw {SYNTH_ZIP}


In [ ]:
# Resume: bring back the run folders of a previous version of this notebook (Add Input -> Notebook Output).
# Finished runs (status done) are not trained again; paused runs continue from their last checkpoint.
import glob, json, shutil
os.makedirs(OUT, exist_ok=True)
ORIGIN = {}  # run folder -> where it came from
for st in glob.glob('/kaggle/input/**/cls/*/status.json', recursive=True):
    src = os.path.dirname(st)
    dst = f'{OUT}/{os.path.basename(src)}'
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
        ORIGIN[os.path.basename(src)] = src
        print('restored', os.path.basename(src), json.load(open(st)).get('state'), 'from', src)

def show_models():
    """Every F-cls run in OUT: where it comes from, its LoRA adapter and the base model it was fine-tuned from."""
    runs = sorted(glob.glob(f'{OUT}/*/status.json'))
    if not runs:
        print('no F-cls run yet: every run will be trained in this session')
    for st in runs:
        d = os.path.dirname(st)
        name = os.path.basename(d)
        if TAG and not name.endswith('_' + TAG):
            continue
        info = json.load(open(f'{d}/run_info.json')) if os.path.exists(f'{d}/run_info.json') else {}
        base = (info.get('config') or {}).get('model', {}).get('name') or '?'
        acfg = f'{d}/adapter/adapter_config.json'
        if os.path.exists(acfg):
            base = json.load(open(acfg)).get('base_model_name_or_path') or base
            size = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(f'{d}/adapter') for f in fs)
            adapter = f'{d}/adapter ({size / 2**20:.0f} MB)'
        state = json.load(open(st)).get('state')
        if not os.path.exists(acfg):
            adapter = 'no adapter folder (only its probabilities are kept)' if state == 'done' else 'not trained yet'
        print(f'[model] {name}: state={state}')
        print(f'        source : {ORIGIN.get(name, "this session / " + OUT)}')
        print(f'        LoRA   : {adapter}')
        print(f'        base   : {base}  https://huggingface.co/{base}  revision={info.get("model_revision")}')

show_models()


In [ ]:
# E2 test suites: the local Qwen2.5-Coder-7B (base, not fine-tuned) reads every gold / public exam (statement,
# prototypes, data file) and writes reference solutions, inputs and a driver; each test keeps the answer most
# references agree on (src/task2/test_gen.py). Runs once per exam on the GPU; kept in /kaggle/working/testgen and
# restored from a previous version (Add Input), so a rerun does not generate again. The records below summarise every
# submission's runs on its exam's suite for F-cls (synthetic exams use the problem bank's own tests).
TESTGEN = '/kaggle/working/testgen'
for idx in glob.glob('/kaggle/input/**/testgen/index.json', recursive=True):
    if not os.path.exists(f'{TESTGEN}/index.json'):
        shutil.copytree(os.path.dirname(idx), TESTGEN, dirs_exist_ok=True)
        print('restored test suites from', os.path.dirname(idx))
print('[model] test generator: Qwen/Qwen2.5-Coder-7B-Instruct (base, not fine-tuned)  '
      'https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct  (revision: see the [testgen] model line)')
!python scripts/gen_task2_tests.py --backend hf --sets gold public --cache {TESTGEN}


In [ ]:
# Records: compile + run on the statement examples and on the exam's test suite (g++), gold mutants, leakage guard.
import json
!python scripts/build_task2_cls_data.py --config {CONFIG} --suites {TESTGEN}/index.json
summary = json.load(open('data/sft/task2_cls/summary.json'))
os.makedirs(OUT, exist_ok=True)
shutil.copy('data/sft/task2_cls/summary.json', f'{OUT}/data_summary.json')
print(json.dumps({k: summary[k] for k in ('n', 'by_source', 'folds', 'empty', 'compiler')}, ensure_ascii=False, indent=1))
assert summary['compiler'], 'no compiler: compile_log would be missing (train/serve mismatch)'


In [ ]:
# Train: the 2 folds in parallel (one GPU each), then the final model. Every run stops cleanly before DEADLINE.
import sys
DEADLINE = T0 + SESSION_HOURS * 3600
def cmd(holdout):
    c = [sys.executable, 'scripts/train_task2_cls.py', '--config', CONFIG, '--holdout', holdout, '--out-root', OUT,
         '--deadline', str(DEADLINE), '--resume-dir', '/kaggle/input']
    if HUB_REPO: c += ['--hub-repo', HUB_REPO]
    if TAG: c += ['--tag', TAG]
    if EPOCHS: c += ['--epochs', str(EPOCHS)]
    if LR: c += ['--lr', str(LR)]
    if SMOKE: c += ['--limit-train', '16', '--epochs', '1']
    return c
def launch(holdout, gpu):
    log = open(f'{OUT}/{holdout}{"_" + TAG if TAG else ""}.log', 'a')
    commit = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
    log.write(f'===== run start {time.strftime("%Y-%m-%d %H:%M:%S")} | {commit} =====\n'); log.flush()
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu), 'PYTHONUNBUFFERED': '1',
           'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}  # less fragmentation (OOM fix)
    return subprocess.Popen(cmd(holdout), stdout=log, stderr=subprocess.STDOUT, env=env)
def tail(holdout, n=5):
    p = f'{OUT}/{holdout}{"_" + TAG if TAG else ""}.log'
    print(f'--- {holdout}:', *open(p).read().splitlines()[-n:], sep='\n  ')
import torch
n_gpu = torch.cuda.device_count()
codes = {}
if n_gpu >= 2:
    procs = {'EX01': launch('EX01', 0), 'EX02': launch('EX02', 1)}
    while any(p.poll() is None for p in procs.values()):
        time.sleep(300)
        for h in procs: tail(h, 2)
    codes.update({h: p.returncode for h, p in procs.items()})
else:
    for h in ('EX01', 'EX02'):
        codes[h] = launch(h, 0).wait()
codes['all'] = launch('all', 0).wait()
for h in codes: tail(h)
print('exit codes (0 = done, 3 = paused by the deadline):', codes)
PAUSED = [h for h, c in codes.items() if c == 3]
FAILED = [h for h, c in codes.items() if c not in (0, 3)]
if PAUSED:
    print(f'PAUSED {PAUSED}: Add Input -> this notebook\'s latest version output, then Save & Run All again (resumes).')
assert not FAILED, f'runs failed: {FAILED} — see {OUT}/<holdout>.log'


In [ ]:
# Out-of-fold scores on gold, thresholds, public predictions (only when every run is done)
if not PAUSED:
    !python scripts/eval_task2_cls.py --config {CONFIG} --out-root {OUT} {('--tag ' + TAG) if TAG else ''}
    import glob
    rep = glob.glob(f'{OUT}/*_report{"_" + TAG if TAG else ""}')[0]
    if os.path.exists(f'{rep}/predictions.json'):
        shutil.copy(f'{rep}/predictions.json', '/kaggle/working/task2_predictions.json')
        print('-> /kaggle/working/task2_predictions.json')
    from IPython.display import Markdown, display
    display(Markdown(open(f'{rep}/report.md').read()))


In [ ]:
# E2: every public submission runs on its exam's generated tests (g++, suites of the cell above), the teacher
# rules label it, and the F-cls predictions above give the causes of wrong answers / the labels where there is no
# test signal.
if not PAUSED:
    show_models()  # the fine-tuned F-cls runs whose predictions are combined below
    print('[model] F-cls predictions:', f'{rep}/predictions.json')
    EXEC = f'{OUT}/exec2{"_" + TAG if TAG else ""}'
    !python scripts/run_task2_exec.py --model-preds {rep}/predictions.json --ids {rep}/predictions.json --suites {TESTGEN}/index.json --out {EXEC}
    shutil.copy(f'{EXEC}/predictions.json', '/kaggle/working/task2_predictions_exec.json')
    print('-> /kaggle/working/task2_predictions_exec.json (submit this one)')
    display(Markdown(open(f'{EXEC}/report.md').read()))


In [ ]:
# Space used by checkpoints (keep /kaggle/working under 20 GB)
!du -sh {OUT}/* 2>/dev/null | sort -h | tail
